# 04 — Inference

This notebook runs the trained model in inference mode for a specific
`now` timestamp, generating forecast-error corrections for every station
in your bounding box.

**Steps**
1. Pick a `now` timestamp (or use the current UTC hour).
2. Run inference for every forecast hour in `config.yaml`.
3. Visualise predictions on an interactive station map.
4. (BNN only) Show separate maps for epistemic and aleatoric uncertainty.

**Prerequisites**: run `01_setup_data.ipynb` and `02_train.ipynb` first.
This notebook writes one parquet per `(fh, metvar)` to
`./output/results/fh{N}_{var}_inference_out.parquet`.

In [ ]:
import sys, os

REPO_ROOT = os.path.abspath("../..")
APP_ROOT  = os.path.abspath("..")
for p in (REPO_ROOT, APP_ROOT):
    if p not in sys.path:
        sys.path.insert(0, p)

os.chdir(APP_ROOT)
print("Working directory:", os.getcwd())

In [ ]:
from app.utils.config_loader import load_config

cfg = load_config()
print("Model type      :", cfg.model)
print("Forecast hours  :", cfg.data.forecast_hours)
print("Met variables   :", cfg.data.metvars)
print("Results dir     :", cfg.output.results_dir)

## Step 1 — Choose `now`

Set `now` to any past timestamp that:
- Falls within your data range (between `start_date` and `end_date`).
- Has HRRR forecasts and ASOS observations available in `./output/parquets/`.

Leave `now = None` to default to the most recent UTC hour.

In [ ]:
from datetime import datetime, timezone, timedelta

# ---- Set your desired inference time here ----
# Example: now = datetime(2024, 6, 15, 18, 0, 0)
now = None
# ----------------------------------------------

if now is None:
    now = datetime.now(tz=timezone.utc).replace(
        minute=0, second=0, microsecond=0, tzinfo=None
    )
else:
    now = now.replace(tzinfo=None)

print(f"Running inference for: {now}  UTC")

## Step 2 — Set up the inference environment

In [ ]:
from pathlib import Path
from app.utils.engine_bridge import (
    prepare_env,
    make_station_climdiv_pickle,
    patch_lookup_parquet,
)

# Set model and output directory env vars (must happen before engine imports)
prepare_env(cfg)

# Write the station → climate-division pickle that the inference engine loads.
pkl_path = make_station_climdiv_pickle(cfg)
print("station_to_climdiv pickle written to:", pkl_path)

# Redirect the station-cluster lookup.
patch_lookup_parquet(cfg)

print("Environment ready.")

## Step 3 — Run inference

In [ ]:
import importlib
from app.utils.engine_bridge import (
    patch_rapids_data_loaders,
    patch_data_loaders,
    patch_inference_pickle,
)

# Select the right engine based on config.
engine_map = {
    "lstm":   "src.lstm_s2s_engine",
    "bnn":    "src.bnn_s2s_engine",
    "hybrid": "src.hybrid_s2s_engine",
}
engine_name = engine_map.get(cfg.model)
if engine_name is None:
    raise ValueError(f"Unknown model type '{cfg.model}'.")

# Run inference under the data-loader and pickle patches.
with patch_rapids_data_loaders(cfg), patch_data_loaders(cfg), patch_inference_pickle(cfg):
    engine = importlib.import_module(engine_name)

    for fh in cfg.data.forecast_hours:
        print(f"\n  fh={fh} …", end="", flush=True)
        if cfg.model == "bnn":
            engine.main(now, fh, mc_samples=cfg.training.mc_samples)
        else:
            engine.main(now, fh)
        print(" done.")

print("\nInference complete.  Results written to:", cfg.output.results_dir)

## Step 4 — Load predictions and display on a map

In [ ]:
import pandas as pd
import folium

results_dir = Path(cfg.output.results_dir)
meta_path   = Path(cfg.output.models_dir) / "lookups" / "station_meta.csv"
station_meta = pd.read_csv(meta_path) if meta_path.exists() else pd.DataFrame()

fh_plot  = cfg.data.forecast_hours[0]
var_plot = cfg.data.metvars[0]

pq_path = results_dir / f"fh{fh_plot}_{var_plot}_inference_out.parquet"
if not pq_path.exists():
    print(f"No predictions found at {pq_path}")
else:
    preds = pd.read_parquet(pq_path).reset_index()
    preds["valid_time"] = pd.to_datetime(preds["valid_time"])
    snap = preds[preds["valid_time"] == pd.Timestamp(now)]

    if snap.empty:
        # Fallback: show the most recent available row
        snap = preds[preds["valid_time"] == preds["valid_time"].max()]
        print(f"No predictions for {now} — showing most recent: {snap['valid_time'].iloc[0]}")

    station_col = "stid" if "stid" in snap.columns else "station"

    # Scalar model output — take step-0 if stored as a list
    def _scalar(v):
        if hasattr(v, '__getitem__'):
            return float(v[0])
        return float(v)

    snap = snap.copy()
    snap["pred_scalar"] = snap["model_output"].apply(_scalar)

    # Merge station lat/lon
    if not station_meta.empty:
        snap = snap.merge(
            station_meta.rename(columns={"station": station_col}),
            on=station_col, how="left",
        )

    centre_lat = (cfg.bbox.lat_min + cfg.bbox.lat_max) / 2
    centre_lon = (cfg.bbox.lon_min + cfg.bbox.lon_max) / 2
    m = folium.Map(location=[centre_lat, centre_lon], zoom_start=7,
                   tiles="CartoDB positron")

    vals = snap["pred_scalar"].dropna()
    vmin, vmax = vals.min(), vals.max()
    span = max(vmax - vmin, 1e-6)

    import matplotlib.cm as cm
    import matplotlib.colors as mcolors
    cmap = cm.get_cmap("RdBu_r")

    for _, row in snap.dropna(subset=["lat", "lon", "pred_scalar"]).iterrows():
        frac = (row["pred_scalar"] - vmin) / span
        rgba = cmap(frac)
        colour = mcolors.to_hex(rgba)
        folium.CircleMarker(
            location=[row.lat, row.lon],
            radius=7,
            color=colour,
            fill=True,
            fill_color=colour,
            fill_opacity=0.85,
            tooltip=(
                f"{row[station_col]} | fh={fh_plot} {var_plot} | "
                f"predicted error = {row['pred_scalar']:.4f}"
            ),
        ).add_to(m)

    print(f"Showing fh={fh_plot}, metvar={var_plot}, valid_time={snap['valid_time'].iloc[0]}")
    display(m)

## Step 5 — BNN uncertainty maps (BNN model only)

In [ ]:
if cfg.model == "bnn":
    import folium
    import matplotlib.cm as cm
    import matplotlib.colors as mcolors

    preds_bnn = pd.read_parquet(pq_path).reset_index()
    preds_bnn["valid_time"] = pd.to_datetime(preds_bnn["valid_time"])
    snap_bnn = preds_bnn[preds_bnn["valid_time"] == pd.Timestamp(now)]
    if snap_bnn.empty:
        snap_bnn = preds_bnn[preds_bnn["valid_time"] == preds_bnn["valid_time"].max()]

    station_col = "stid" if "stid" in snap_bnn.columns else "station"
    if not station_meta.empty:
        snap_bnn = snap_bnn.merge(
            station_meta.rename(columns={"station": station_col}),
            on=station_col, how="left",
        )

    def _scalar(v):
        if hasattr(v, '__getitem__'):
            return float(v[0])
        return float(v)

    for unc_col, title in [
        ("epistemic_var", "Epistemic uncertainty (model)"),
        ("aleatoric_var", "Aleatoric uncertainty (data)"),
    ]:
        if unc_col not in snap_bnn.columns:
            print(f"Column '{unc_col}' not found — skipping.")
            continue

        snap_bnn[f"{unc_col}_s"] = snap_bnn[unc_col].apply(_scalar)
        vals = snap_bnn[f"{unc_col}_s"].dropna()
        vmin, vmax = vals.min(), vals.max()
        span = max(vmax - vmin, 1e-9)

        mu = folium.Map(location=[centre_lat, centre_lon], zoom_start=7,
                        tiles="CartoDB positron")
        cmap_unc = cm.get_cmap("viridis")

        for _, row in snap_bnn.dropna(subset=["lat", "lon", f"{unc_col}_s"]).iterrows():
            frac = (row[f"{unc_col}_s"] - vmin) / span
            colour = mcolors.to_hex(cmap_unc(frac))
            folium.CircleMarker(
                location=[row.lat, row.lon],
                radius=7,
                color=colour,
                fill=True,
                fill_color=colour,
                fill_opacity=0.85,
                tooltip=(
                    f"{row[station_col]} | {unc_col} (step 0) = "
                    f"{row[f'{unc_col}_s']:.4e}"
                ),
            ).add_to(mu)

        print(f"\n{title} — fh={fh_plot} {var_plot}")
        display(mu)
else:
    print(f"BNN uncertainty maps are only available when model='bnn' (current: '{cfg.model}').")

## Step 6 — Inspect raw output

The table below shows the raw inference-output parquet for the current
`now` timestamp and the first (fh, metvar) combination.

In [ ]:
pq_path = Path(cfg.output.results_dir) / f"fh{fh_plot}_{var_plot}_inference_out.parquet"
if pq_path.exists():
    df_out = pd.read_parquet(pq_path).reset_index()
    df_out["valid_time"] = pd.to_datetime(df_out["valid_time"])
    snap = df_out[df_out["valid_time"] == pd.Timestamp(now)]
    if snap.empty:
        snap = df_out[df_out["valid_time"] == df_out["valid_time"].max()]
    display(snap.head(20))
else:
    print("No output file found.")